# Longor M3 launcher (`libero_10` task 0)

Runs the full Longor path (planner + executor + VLA service) on `libero_10` task 0.

Goal for M3:
- At least one successful end-to-end episode (`success=True` in run summary).

Notes:
- Uses current runtime default `VLA_ACTIONS_PER_CHUNK` from `src/config.py`.
- Section 7 streams full stdout/stderr live from a single run.


## 1 — Clone repo and capture secrets

In [ ]:
import getpass

github_token = getpass.getpass('GitHub PAT (repo scope): ')
openai_api_key = getpass.getpass('OpenAI API key: ')

In [ ]:
import os

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

## 2 — Headless rendering + app deps

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get update
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r deps/requirements-libero.txt
%pip install -e .

# Silence robosuite macro recommendation warning for this runtime.
!python -m robosuite.scripts.setup_macros

## 3 — Pi0.5 (OpenPI) venv

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
VLA_VENV=/content/Longor/.vla-venv
OPENPI_DIR=/content/openpi
VLA_PYTHON="$VLA_VENV/bin/python"

python -m pip install --quiet virtualenv
[ -d "$VLA_VENV" ] || python -m virtualenv "$VLA_VENV"
"$VLA_PYTHON" -m pip install -U pip

[ -d "$OPENPI_DIR" ] || git clone --recurse-submodules https://github.com/Physical-Intelligence/openpi "$OPENPI_DIR"
"$VLA_PYTHON" -m pip install -e "$OPENPI_DIR"
"$VLA_PYTHON" -m pip install -r deps/requirements-vla-pi05-linux.txt
"$VLA_PYTHON" -m pip install -e .

## 4 — Configure `.env` for Longor run

In [ ]:
from pathlib import Path

Path('.env').write_text(
    f'OPENAI_API_KEY={openai_api_key}\n'
    'LOG_LEVEL=DEBUG\n'
    'LLM_MODEL=gpt-4.1-mini\n'
    'SAVE_VIDEO=1\n',
    encoding='utf-8',
)
print('.env written')

## 5 — Start VLA server + health

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
VLA_LOG=/tmp/longor_m3_vla_server.log
VLA_PYTHON=/content/Longor/.vla-venv/bin/python

cat >/tmp/longor_m3.boto <<'EOF'
[GSUtil]
check_hashes = never
EOF

export BOTO_CONFIG=/tmp/longor_m3.boto
export PYTHONPATH="/content/openpi:/content/openpi/src:/content/openpi/packages/openpi/src:${PYTHONPATH:-}"

pkill -f "services/run_vla_server.py" || true
nohup "$VLA_PYTHON" services/run_vla_server.py >"$VLA_LOG" 2>&1 &

for _ in $(seq 1 360); do
  if curl -fsS http://localhost:8765/health; then
    echo
    echo "VLA server healthy"
    exit 0
  fi
  sleep 5
done

echo "VLA server not healthy; see $VLA_LOG" >&2
exit 1

## 6 — Run Longor once (`libero_10`, task 0)


In [ ]:
!python examples/long_horizon_llm_libero.py

## 7 — Save the video

### HOW TO GET VIDEO FILES (when running via Colab extension)?
1. run with 'SAVE_VIDEO=1'
2. click Colab -> mount server to workspace
3. reload IDE window.
4. open the files sidebar! :)
